In [ ]:

# ==============================================================
# IITM TDS: MORPH TWO FACES INTO A VIDEO
# Complete Google Colab program - run in ONE cell
# Supports image upload OR existing file paths
# Output: /content/face_morph.mp4 (under 100,000 bytes)
# ==============================================================

import os
import sys
import subprocess
import urllib.request
import importlib.util

# ---------------- USER SETTINGS ----------------

INPUT_MODE = "upload"  # Choose "upload" or "path"

# Used only when INPUT_MODE = "path"
IMAGE_A_PATH = "/content/face_a.jpg"  # First face
IMAGE_B_PATH = "/content/face_b.jpg"  # Second face

OUTPUT_PATH = "/content/face_morph.mp4"

# Video settings
SIZE = 160
FPS = 10
DURATION = 3
MAX_BYTES = 100000

# ------------------------------------------------
# 1. Install/check required packages
# ------------------------------------------------

print("Step 1/7: Checking Python packages...")

def ensure_package(import_name, package_name):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "-q", package_name
        ])

ensure_package("cv2", "opencv-python-headless")
ensure_package("mediapipe", "mediapipe")
ensure_package("scipy", "scipy")

import cv2
import numpy as np
import mediapipe as mp
from scipy.spatial import Delaunay
from google.colab import files
from IPython.display import display, Video

# Verify that the newer MediaPipe Tasks API exists
if not hasattr(mp, "tasks"):
    raise RuntimeError(
        "This MediaPipe installation does not expose the Tasks API. "
        "Restart the Colab session and run this cell again. "
        "If it persists, send me the MediaPipe version and error."
    )

# ------------------------------------------------
# 2. Upload photos or load them from file paths
# ------------------------------------------------

print("\nStep 2/7: Loading photographs...")

def read_image(path):
    image = cv2.imread(path)
    if image is None:
        raise ValueError(
            f"Could not open: {path}\n"
            "Check the filename and path."
        )
    return image

if INPUT_MODE.lower() == "upload":
    print("Choose the FIRST face photograph.")
    uploaded_a = files.upload()
    if not uploaded_a:
        raise RuntimeError("No first photograph was uploaded.")

    name_a = next(iter(uploaded_a))
    with open("/content/face_a_upload", "wb") as f:
        f.write(uploaded_a[name_a])
    image_a = read_image("/content/face_a_upload")

    print("\nChoose the SECOND face photograph.")
    uploaded_b = files.upload()
    if not uploaded_b:
        raise RuntimeError("No second photograph was uploaded.")

    name_b = next(iter(uploaded_b))
    with open("/content/face_b_upload", "wb") as f:
        f.write(uploaded_b[name_b])
    image_b = read_image("/content/face_b_upload")

elif INPUT_MODE.lower() == "path":
    image_a = read_image(IMAGE_A_PATH)
    image_b = read_image(IMAGE_B_PATH)

else:
    raise ValueError('INPUT_MODE must be "upload" or "path".')

print("First image:", image_a.shape)
print("Second image:", image_b.shape)

# ------------------------------------------------
# 3. Prepare images and download landmark model
# ------------------------------------------------

print("\nStep 3/7: Preparing face detection...")

def prepare(image):
    h, w = image.shape[:2]
    scale = min(SIZE / w, SIZE / h)
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    resized = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.full((SIZE, SIZE, 3), 245, dtype=np.uint8)
    x = (SIZE - nw) // 2
    y = (SIZE - nh) // 2
    canvas[y:y+nh, x:x+nw] = resized
    return canvas

A = prepare(image_a)
B = prepare(image_b)

MODEL_PATH = "/content/face_landmarker.task"
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/"
    "face_landmarker.task"
)

if not os.path.exists(MODEL_PATH):
    print("Downloading the facial-landmark model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

# ------------------------------------------------
# 4. Detect corresponding facial landmarks
# ------------------------------------------------

print("\nStep 4/7: Detecting faces...")

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=RunningMode.IMAGE,
    num_faces=1
)

def detect(image, landmarker):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )
    result = landmarker.detect(mp_image)

    if not result.face_landmarks:
        raise RuntimeError(
            "No face detected in one of the photos. "
            "Use clear photographs showing the full face."
        )

    return np.array([
        [p.x * SIZE, p.y * SIZE]
        for p in result.face_landmarks[0]
    ], dtype=np.float32)

with FaceLandmarker.create_from_options(options) as landmarker:
    PA = detect(A, landmarker)
    PB = detect(B, landmarker)

if len(PA) != len(PB):
    raise RuntimeError("The two photos produced different landmark counts.")

# Add corner points to cover the image edges
corners = np.array([
    [0, 0], [SIZE-1, 0],
    [SIZE-1, SIZE-1], [0, SIZE-1]
], dtype=np.float32)

PA = np.vstack([PA, corners])
PB = np.vstack([PB, corners])

# Create a shared triangle layout
triangles = Delaunay((PA + PB) / 2).simplices

print("Detected", len(PA)-4, "facial landmarks in each photo.")

# ------------------------------------------------
# 5. Warp and blend faces for each video frame
# ------------------------------------------------

print("\nStep 5/7: Creating morph animation...")

def warp_image(source, source_points, destination_points):
    output = np.zeros_like(source)

    for tri in triangles:
        src = source_points[tri]
        dst = destination_points[tri]

        x1, y1, w1, h1 = cv2.boundingRect(
            np.float32([src])
        )
        x2, y2, w2, h2 = cv2.boundingRect(
            np.float32([dst])
        )

        if min(w1, h1, w2, h2) <= 0:
            continue

        crop = source[y1:y1+h1, x1:x1+w1]
        if crop.size == 0:
            continue

        src_local = src - np.array([x1, y1], dtype=np.float32)
        dst_local = dst - np.array([x2, y2], dtype=np.float32)

        matrix = cv2.getAffineTransform(
            np.float32(src_local), np.float32(dst_local)
        )

        warped = cv2.warpAffine(
            crop, matrix, (w2, h2),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT_101
        )

        mask = np.zeros((h2, w2), dtype=np.uint8)
        cv2.fillConvexPoly(
            mask, np.int32(dst_local), 255,
            lineType=cv2.LINE_AA
        )

        roi = output[y2:y2+h2, x2:x2+w2]
        output[y2:y2+h2, x2:x2+w2] = np.where(
            mask[:, :, None] > 0, warped, roi
        )

    return output

frame_count = FPS * DURATION
frames = []

for i in range(frame_count):
    linear_t = i / max(frame_count - 1, 1)

    # Ease-in/ease-out gives a gentler transition
    t = linear_t * linear_t * (3 - 2 * linear_t)

    intermediate_points = (1-t) * PA + t * PB

    warped_a = warp_image(A, PA, intermediate_points)
    warped_b = warp_image(B, PB, intermediate_points)

    blended = (
        (1-t) * warped_a.astype(np.float32)
        + t * warped_b.astype(np.float32)
    )

    frames.append(np.clip(blended, 0, 255).astype(np.uint8))

# ------------------------------------------------
# 6. Encode and compress below 100,000 bytes
# ------------------------------------------------

print("\nStep 6/7: Encoding and compressing video...")

if subprocess.run(
    ["ffmpeg", "-version"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
).returncode != 0:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "ffmpeg"],
        check=True
    )

RAW_PATH = "/content/face_morph_raw.mp4"

writer = cv2.VideoWriter(
    RAW_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (SIZE, SIZE)
)

if not writer.isOpened():
    raise RuntimeError("Could not open the video writer.")

for frame in frames:
    writer.write(frame)

writer.release()

# Try progressively smaller/stronger compression settings.
# Do not truncate the video to meet the limit.
settings = [
    (160, 10, 38),
    (128, 8, 42),
    (112, 7, 45),
    (96, 6, 48),
    (80, 5, 50),
    (64, 5, 52)
]

success = False

for width, fps, crf in settings:
    if os.path.exists(OUTPUT_PATH):
        os.remove(OUTPUT_PATH)

    command = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", RAW_PATH,
        "-vf", f"scale={width}:{width}:flags=lanczos,fps={fps}",
        "-an",
        "-c:v", "libx264",
        "-preset", "veryslow",
        "-crf", str(crf),
        "-pix_fmt", "yuv420p",
        "-threads", "1",
        "-movflags", "+faststart",
        OUTPUT_PATH
    ]

    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("FFmpeg could not encode the video.")

    size = os.path.getsize(OUTPUT_PATH)
    print(f"{width}px, {fps} FPS: {size:,} bytes")

    if size < MAX_BYTES:
        success = True
        break

if not success:
    raise RuntimeError(
        "The video is still too large. Increase compression "
        "or reduce duration/resolution, then run the cell again."
    )

print("\nSUCCESS!")
print("Output:", OUTPUT_PATH)
print("File size:", f"{os.path.getsize(OUTPUT_PATH):,} bytes")
print("Under 100,000 bytes:", os.path.getsize(OUTPUT_PATH) < MAX_BYTES)

# ------------------------------------------------
# 7. Preview and download
# ------------------------------------------------

print("\nStep 7/7: Previewing your video...")
display(Video(OUTPUT_PATH, embed=True))

print("Downloading the completed video...")
files.download(OUTPUT_PATH)

print(
    "\nNext: upload face_morph.mp4 to a public HTTPS host "
    "and submit its direct video URL to your assignment."
)
